In [1]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

Cloning into 'capstone_project_GroupA'...
remote: Enumerating objects: 1209, done.
remote: Counting objects: 100% (325/325), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 1209 (delta 191), reused 184 (delta 88), pack-reused 884 (from 3)
Receiving objects: 100% (1209/1209), 331.48 MiB | 39.99 MiB/s, done.
Resolving deltas: 100% (597/597), done.
Updating files: 100% (99/99), done.


In [2]:
%cd capstone_project_GroupA
!git checkout colab
%cd src

/content/capstone_project_GroupA
Branch 'colab' set up to track remote branch 'colab' from 'origin'.
Switched to a new branch 'colab'
/content/capstone_project_GroupA/src


In [ ]:
from datetime import datetime
from ModelFiles.GroupAModels import TransformersModel
from ModelFiles.ModelConfigs import TransformersConfig, HORIZONS, SEEDS
from ModelFiles.ModelEnums import TransformerModelType
from ModelFiles.ModelPlots import *

USE_LOG_TARGET = True
CONTEXT_LENGTHS = [48, 336, 720]
EVAL_STEP_SIZE = 48
NUM_EPOCHS = 100
PATIENCE = 10
DEBUG = False

for horizon in HORIZONS:
    for context_length in CONTEXT_LENGTHS:
        if context_length >= horizon:
            for seed in SEEDS:
                if seed == SEEDS[-1]:
                    save_prediction_results = True
                else:
                    save_prediction_results = False

                timexer_config = TransformersConfig(
                    task_id=f"timexer_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                    model=TransformerModelType.TIMEXER,
                    forecast_horizon=horizon,
                    lookback_window=context_length,
                    used_log_target=USE_LOG_TARGET,
                    target_col="LOG_TOTALDEMAND" if USE_LOG_TARGET else "TOTALDEMAND",
                    feature_cols=['TEMPERATURE', 'TEMP_SQUARED', 'IS_WEEKEND', 'demand_1_year_ago'],
                    scale=True,
                    date_col='DATETIME',
                    variate='MS',
                    patch_len=16,
                    stride=16,  # TimeXer uses patch_len as stride internally
                    d_model=512,
                    num_attention_heads=8,
                    num_encoder_layers=3,
                    dim_ff=2048,
                    dropout=0.1,
                    dropout_head_fc=0.1,
                    use_gpu=True,
                    time_encoding='timeF',
                    training_epochs=NUM_EPOCHS,
                    batch_size=32,
                    learning_rate=0.0001,
                    output_attention=False,
                    lradj='type1',
                    patience=PATIENCE,
                    seed=seed,
                    eval_step_size=EVAL_STEP_SIZE,
                    save_test_results=save_prediction_results,
                    debug=DEBUG,
                    save_training_log=True,
                    use_norm=True,
                    activation='gelu',
                )
                timexer_model = TransformersModel(timexer_config)
                timexer_model.train_model()
                timexer_model.evaluate_model(test_mode=1)
                print("=" * 200)
                print("\n")


Found NSW data path: /content/capstone_project_GroupA/data/NSW


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Set random seed to 31415
Use GPU: cuda:0
train 107300
	iters: 100, epoch: 1 | loss: 0.2150693
	speed: 0.0510s/iter; left time: 17078.9290s
	iters: 200, epoch: 1 | loss: 0.2492466
	speed: 0.0200s/iter; left time: 6691.5602s
